# Section 3 — ELT with WebHDFS `InsecureClient`
**Big Data with Hadoop · Palmer Penguins Assessment**

Connect to the Hadoop sandbox over **WebHDFS**, load `penguins.csv` from HDFS into Pandas, analyse structure and missing values, clean it, convert body mass to kilograms, and upload the result back to HDFS as `/user/maria_dev/datasets/clean/penguins_clean.csv`.

In [1]:
# pip install hdfs pandas
import io
import pandas as pd
from hdfs import InsecureClient

# WebHDFS endpoint of the sandbox NameNode (HDP sandbox: http://sandbox-hdp.hortonworks.com:50070)
WEBHDFS_URL = "http://localhost:9870"
HDFS_USER   = "maria_dev"
RAW_PATH    = "/user/maria_dev/datasets/penguins.csv"
CLEAN_PATH  = "/user/maria_dev/datasets/clean/penguins_clean.csv"

client = InsecureClient(WEBHDFS_URL, user=HDFS_USER)
print("Connected to:", WEBHDFS_URL, "as", HDFS_USER)
print(client.list("/user/maria_dev/datasets"))

Connected to: http://localhost:9870 as maria_dev
['penguins.csv']


## 1. Load `penguins.csv` from HDFS into a Pandas DataFrame

In [2]:
status = client.status(RAW_PATH)
print(f"HDFS file size: {status['length']} bytes | replication: {status['replication']} | owner: {status['owner']}")

with client.read(RAW_PATH, encoding="utf-8") as reader:
    df = pd.read_csv(reader, na_values=["NA", ""])

df.head(10)

HDFS file size: 15241 bytes | replication: 1 | owner: maria_dev


,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,year
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,male,2007
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,female,2007
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,female,2007
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN,2007
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,female,2007
5,Adelie,Torgersen,39.3,20.6,190.0,3650.0,male,2007
6,Adelie,Torgersen,38.9,17.8,181.0,3625.0,female,2007
7,Adelie,Torgersen,39.2,19.6,195.0,4675.0,male,2007
8,Adelie,Torgersen,34.1,18.1,193.0,3475.0,NaN,2007
9,Adelie,Torgersen,42.0,20.2,190.0,4250.0,NaN,2007


## 2. Analyse structure

In [3]:
print("Shape (rows, columns):", df.shape)
print()
df.info()

Shape (rows, columns): (344, 8)

<class 'pandas.DataFrame'>
RangeIndex: 344 entries, 0 to 343
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   species            344 non-null    str    
 1   island             344 non-null    str    
 2   bill_length_mm     342 non-null    float64
 3   bill_depth_mm      342 non-null    float64
 4   flipper_length_mm  342 non-null    float64
 5   body_mass_g        342 non-null    float64
 6   sex                333 non-null    str    
 7   year               344 non-null    int64  
dtypes: float64(4), int64(1), str(3)
memory usage: 21.6 KB


In [4]:
df.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
species,344,3,Adelie,152,NaN,NaN,NaN,NaN,NaN,NaN,NaN
island,344,3,Biscoe,168,NaN,NaN,NaN,NaN,NaN,NaN,NaN
bill_length_mm,342.0,NaN,NaN,NaN,43.92193,5.459584,32.1,39.225,44.45,48.5,59.6
bill_depth_mm,342.0,NaN,NaN,NaN,17.15117,1.974793,13.1,15.6,17.3,18.7,21.5
flipper_length_mm,342.0,NaN,NaN,NaN,200.915205,14.061714,172.0,190.0,197.0,213.0,231.0
body_mass_g,342.0,NaN,NaN,NaN,4201.754386,801.954536,2700.0,3550.0,4050.0,4750.0,6300.0
sex,333,2,male,168,NaN,NaN,NaN,NaN,NaN,NaN,NaN
year,344.0,NaN,NaN,NaN,2008.02907,0.818356,2007.0,2007.0,2008.0,2009.0,2009.0


In [5]:
print(df['species'].value_counts(), "\n")
print(df['island'].value_counts(), "\n")
print(df['sex'].value_counts(dropna=False))

species
Adelie       152
Gentoo       124
Chinstrap     68
Name: count, dtype: int64 

island
Biscoe       168
Dream        124
Torgersen     52
Name: count, dtype: int64 

sex
male      168
female    165
NaN        11
Name: count, dtype: int64


## 3. Missing values

In [6]:
missing = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_pct":   (df.isna().mean() * 100).round(2)
})
missing

,missing_count,missing_pct
species,0,0.00
island,0,0.00
bill_length_mm,2,0.58
bill_depth_mm,2,0.58
flipper_length_mm,2,0.58
body_mass_g,2,0.58
sex,11,3.20
year,0,0.00


In [7]:
# Rows where every measurement is missing carry no information
measure_cols = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
df[df[measure_cols].isna().all(axis=1)]

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,year
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN,2007
271,Gentoo,Biscoe,NaN,NaN,NaN,NaN,NaN,2009


## 4. Clean & preprocess
1. Drop rows where **all** body measurements are missing (2 rows).
2. Impute any remaining numeric gaps with the **species median**.
3. Normalise `sex` to lower case; fill missing sex with `"unknown"`.
4. Convert **body mass to kilograms** (`body_mass_kg = body_mass_g / 1000`).
5. Remove duplicates and enforce data types.

In [8]:
clean = df.copy()
before = len(clean)

# 1. drop rows with no measurements at all
clean = clean.dropna(subset=measure_cols, how="all")

# 2. species-median imputation for numeric columns
for col in measure_cols:
    clean[col] = clean.groupby("species")[col].transform(lambda s: s.fillna(s.median()))

# 3. tidy sex
clean["sex"] = clean["sex"].str.strip().str.lower().fillna("unknown")

# 4. body mass in kilograms
clean["body_mass_kg"] = (clean["body_mass_g"] / 1000).round(3)

# 5. duplicates + dtypes
clean = clean.drop_duplicates()
clean["flipper_length_mm"] = clean["flipper_length_mm"].astype(float)
clean["year"] = clean["year"].astype(int)

clean = clean[["species", "island", "bill_length_mm", "bill_depth_mm",
               "flipper_length_mm", "body_mass_g", "body_mass_kg", "sex", "year"]]

print(f"Rows before: {before} | after: {len(clean)} | dropped: {before - len(clean)}")
print("\nMissing values after cleaning:")
print(clean.isna().sum())
clean.head(10)

Rows before: 344 | after: 342 | dropped: 2

Missing values after cleaning:
species              0
island               0
bill_length_mm       0
bill_depth_mm        0
flipper_length_mm    0
body_mass_g          0
body_mass_kg         0
sex                  0
year                 0
dtype: int64


,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,body_mass_kg,sex,year
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,3.750,male,2007
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,3.800,female,2007
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,3.250,female,2007
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,3.450,female,2007
5,Adelie,Torgersen,39.3,20.6,190.0,3650.0,3.650,male,2007
6,Adelie,Torgersen,38.9,17.8,181.0,3625.0,3.625,female,2007
7,Adelie,Torgersen,39.2,19.6,195.0,4675.0,4.675,male,2007
8,Adelie,Torgersen,34.1,18.1,193.0,3475.0,3.475,unknown,2007
9,Adelie,Torgersen,42.0,20.2,190.0,4250.0,4.250,unknown,2007
10,Adelie,Torgersen,37.8,17.1,186.0,3300.0,3.300,unknown,2007


In [9]:
clean.groupby("species").agg(
    penguins=("species", "size"),
    avg_mass_kg=("body_mass_kg", "mean"),
    avg_flipper_mm=("flipper_length_mm", "mean")
).round(3)

,penguins,avg_mass_kg,avg_flipper_mm
species,,,
Adelie,151,3.701,189.954
Chinstrap,68,3.733,195.824
Gentoo,123,5.076,217.187


## 5. Upload cleaned dataset to HDFS

In [10]:
client.makedirs("/user/maria_dev/datasets/clean")
csv_text = clean.to_csv(index=False)
client.write(CLEAN_PATH, data=csv_text, encoding="utf-8", overwrite=True)

print("Uploaded:", CLEAN_PATH)
print(client.list("/user/maria_dev/datasets/clean", status=True))

Uploaded: /user/maria_dev/datasets/clean/penguins_clean.csv
[('penguins_clean.csv', {'accessTime': 1792660443886, 'blockSize': 134217728, 'childrenNum': 0, 'fileId': 16579, 'group': 'supergroup', 'length': 18357, 'modificationTime': 1792660444027, 'owner': 'maria_dev', 'pathSuffix': 'penguins_clean.csv', 'permission': '644', 'replication': 1, 'storagePolicy': 0, 'type': 'FILE'})]


In [11]:
# Verify by reading it back from HDFS
with client.read(CLEAN_PATH, encoding="utf-8") as reader:
    check = pd.read_csv(reader)
print("Read back from HDFS:", check.shape)
check.head()

Read back from HDFS: (342, 9)


,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,body_mass_kg,sex,year
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,3.75,male,2007
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,3.80,female,2007
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,3.25,female,2007
3,Adelie,Torgersen,36.7,19.3,193.0,3450.0,3.45,female,2007
4,Adelie,Torgersen,39.3,20.6,190.0,3650.0,3.65,male,2007
